# make_programs — attention-head program synthesis

Generates one Python program per attention head, approximating that head's
attention pattern, and emits a single `data/{model_key}_programs.py`.

**Model-agnostic.** Layer/head counts, tokenizer and embeddings are read off the
loaded model. Nothing here is specific to GPT-2, Llama, or Qwen.

**Output contract** — every program is emitted in final form, with no conversion step:

```python
def prog_L{layer}H{head}(sentence: str, tokenizer: PreTrainedTokenizerBase) -> Tuple[str, np.ndarray]
```

returning `(label, (n,n) row-stochastic lower-triangular matrix)`. Names ending in
`_L#H#` satisfy both repo loaders — `write_data.ipynb`'s regex and
`all_experiments_v2.ipynb`'s `load_programs`.

**Scoring is soft IoU throughout** (`min/max`, higher is better), matching
`data/iou_scores_*.csv`. JSD is not used.

**One program per head, unconditionally.** Whether a program covers more than one
head is discovered downstream in `write_data.ipynb` by best-fit — not assumed here.

## 1 — Configuration

In [ ]:
import json, os, random, sys, time
from pathlib import Path
import numpy as np, torch
from transformers import AutoModel, AutoTokenizer

sys.path.insert(0, str(Path.cwd() / "lib"))    # engine lives in code/lib/
import program_synthesis as ps          # the engine; see code/program_synthesis.py

MODEL_ID   = "gpt2"          # any HF causal LM: "Qwen/Qwen3-4B", "meta-llama/Llama-3.2-3B", ...
MODEL_KEY  = "gpt2new"       # -> data/{MODEL_KEY}_programs.py
LLM        = "gpt-5"         # litellm route: "gpt-5", "claude-opus-5", "openai/gpt-4.1", ...
STRATEGY   = "two_pass"      # single | two_pass | alpha_blend | greedy | best_of_k
N_CORPUS   = 200             # sentences extracted
N_SCORE    = 12              # sentences used to score each program
WORKERS    = 8              # concurrent heads; raise for big grids, 1 to debug serially
SEED       = 42

ROOT   = Path.cwd().parent
WORK   = ROOT / "code" / f".synthesis_{MODEL_KEY}"
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModel.from_pretrained(MODEL_ID, attn_implementation="eager").to(DEVICE).eval()

N_LAYERS = model.config.num_hidden_layers
N_HEADS  = model.config.num_attention_heads
N_PROGS  = N_LAYERS * N_HEADS

print(f"{MODEL_ID}: {N_LAYERS} layers x {N_HEADS} heads = {N_PROGS} programs")
print(f"device={DEVICE}  strategy={STRATEGY}  llm={LLM}")

### Sanity check

The head grid drives everything downstream, so verify it against a real forward
pass rather than trusting the config. For GQA models (Qwen3, Llama-3) note that
`head_dim * num_attention_heads != hidden_size` — never derive head_dim by division.

In [ ]:
_probe = tokenizer("The small cat sat on the warm window sill.", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    _out = model(**_probe, output_attentions=True)

assert len(_out.attentions) == N_LAYERS, f"{len(_out.attentions)} attention tensors != {N_LAYERS} layers"
assert _out.attentions[0].shape[1] == N_HEADS, f"{_out.attentions[0].shape[1]} heads != {N_HEADS}"
print(f"OK — attentions: {len(_out.attentions)} x {tuple(_out.attentions[0].shape)}")

cfg = model.config
print(f"hidden_size={cfg.hidden_size}  head_dim={getattr(cfg,'head_dim', None)}  "
      f"kv_heads={getattr(cfg,'num_key_value_heads', N_HEADS)}")
if getattr(cfg, "head_dim", None) and cfg.head_dim * N_HEADS != cfg.hidden_size:
    print(f"  NOTE: GQA — head_dim*heads ({cfg.head_dim*N_HEADS}) != hidden_size ({cfg.hidden_size}).")

## 2 — Corpus

TinyStories, 15–60 words per sentence. The **resolved sentence text** is written to
`sentences.json`, not just a seed — so a rerun is reproducible even if the dataset
version or the shuffling library changes underneath.

In [ ]:
SENT_CACHE = WORK / "sentences.json"

sentences       = ps.load_corpus(SENT_CACHE, n=N_CORPUS, seed=SEED)
score_sentences = sentences[:N_SCORE]

print(f"{len(sentences)} sentences; scoring each program on {len(score_sentences)}")
print("example:", sentences[0][:100])


## 3 — Extraction

One `.npz` per sentence holding `(n_layers, n_heads, seq, seq)` in **fp16**, plus
tokens and the source text. Skip-if-exists makes this resumable.

At large scale this is the memory pressure point: Qwen3-4B is 36x32x64x64 per
sentence, ~19 MB fp32. fp16 halves it; shard per layer if you go bigger.

In [ ]:
data = ps.extract_attention(model, tokenizer, sentences, WORK / "attention", DEVICE)

assert data[0]["attention"].shape[:2] == (N_LAYERS, N_HEADS), \
    f'attention {data[0]["attention"].shape[:2]} != config ({N_LAYERS}, {N_HEADS})'
print(f"attention shape per sentence: {data[0]['attention'].shape}")


---
# 4 — THE PROMPTS
---

The most important part of this notebook. Everything below is rendered from
`program_synthesis.py` so what you read here is byte-identical to what the model
receives.

**Design note.** The helper source injected into the prompt is the *same string*
that gets inlined into the emitted module (`ps.helpers_source(MODEL_ID)`). That
identity is deliberate: in the previous llama3b library the prompt advertised
helpers the emitted module never defined, and 51% of programs died on `NameError`.

### 4a — The helper surface advertised to the model

In [ ]:
HELPERS = ps.helpers_source(MODEL_ID)
print(HELPERS)

### 4b — How an attention matrix is shown to the model

`represent_head` buckets edges by weight **quantile** rather than raw rank, and caps
self-attention and first-token edges per band. Naive top-k returns the diagonal and
column 0 for every head, so heads become indistinguishable; stratifying keeps the
structure that actually separates them.

In [ ]:
_demo = ps.represent_head(data[0]["tokens"], data[0]["attention"][0, 0], data[0]["sentence"])
print(_demo)

### 4c — The initial (pass-1) prompt, with placeholders

In [ ]:
_skeleton = ps.build_initial_prompt(
    head_examples="{{HEAD_EXAMPLES}}   <- 5 stratified examples, see 4b",
    layer="{{LAYER}}", head="{{HEAD}}", model_name="{{MODEL_ID}}",
    helpers="{{HELPERS}}   <- the full source printed in 4a",
    fname="prog_L{{LAYER}}H{{HEAD}}")
print(_skeleton)

### 4d — The same prompt, fully rendered for one real head

In [ ]:
_examples = ps.format_head_examples(data, 0, 0, seed=SEED)
_real = ps.build_initial_prompt(_examples, 0, 0, MODEL_ID, HELPERS)
print(_real)
print("\n" + "="*70)
print(f"rendered length: {len(_real):,} chars")

### 4e — The refinement (pass-2) prompt, with placeholders

In [ ]:
_refine = ps.build_refinement_prompt(
    original_code="{{ORIGINAL_CODE}}   <- the pass-1 program verbatim",
    feedback=[(0.87, "{{COMPARISON_BEST}}  <- under/over-predicted edges, best sentence"),
              (0.31, "{{COMPARISON_WORST}} <- under/over-predicted edges, worst sentence")],
    layer="{{LAYER}}", head="{{HEAD}}", model_name="{{MODEL_ID}}",
    helpers="{{HELPERS}}", fname="prog_L{{LAYER}}H{{HEAD}}")
print(_refine)

---
# End of prompts
---

## 5 — LLM client and improvement strategy

The client is provider-agnostic via **litellm**, so `LLM` may name an OpenAI,
Anthropic, or other route. Responses are cached on disk keyed by
SHA(model, system, prompt) — that is what makes a 1000+ head run resumable.

Strategies are pluggable. Each takes a scored initial program and returns the
program to keep plus every candidate it tried:

| name | method |
|---|---|
| `single` | zero-shot only, no improvement |
| `two_pass` | one refinement on scored feedback, keep the better |
| `alpha_blend` | two_pass, then also try `a*initial + (1-a)*refined` as a real composed program |
| `greedy` | refine repeatedly, keep-if-better, stop on first non-improving round |
| `best_of_k` | k diverse hypotheses, keep the best |

Add a function, register it in `ps.STRATEGIES`, and the driver picks it up unchanged.

In [ ]:
sandbox = ps.Sandbox(HELPERS)
llm     = ps.LLMClient(model=LLM, cache_dir=WORK / "llm_cache")
improve = ps.STRATEGIES[STRATEGY]

print("strategies available:", list(ps.STRATEGIES))
print("selected:", STRATEGY)
print("helpers in sandbox:", sorted(n for n in sandbox.helper_names if not n.startswith("_")))

## 6 — Driver

One program per head, unconditionally. `ps.run_grid` is shared with
`run_synthesis.py`, so the notebook and the CLI cannot drift apart.

**Resumable** — a head whose program file and manifest entry both exist is
skipped, so the run can be interrupted freely and picked back up.

**Concurrent** — `WORKERS` heads are generated in parallel. At 1152 heads
(Qwen3-4B) a serial run is impractical; set `WORKERS = 1` when debugging so
tracebacks stay readable.


In [ ]:
PROG_DIR = WORK / "programs"
MANIFEST = WORK / "manifest.json"
HEADS    = [(l, h) for l in range(N_LAYERS) for h in range(N_HEADS)]

manifest = ps.run_grid(
    heads=HEADS, data=data, score_sentences=score_sentences,
    tokenizer=tokenizer, sandbox=sandbox, llm=llm, helpers=HELPERS,
    model_name=MODEL_ID, strategy=STRATEGY,
    prog_dir=PROG_DIR, manifest_path=MANIFEST,
    seed=SEED, workers=WORKERS)

ok  = {k: v for k, v in manifest.items() if v.get("status") == "ok"}
bad = {k: v for k, v in manifest.items() if v.get("status") != "ok"}
print(f"\n{len(ok)}/{len(HEADS)} heads generated; {len(bad)} failed")
for k, v in list(bad.items())[:10]:
    print(f"  {k}: {v.get('status')} — {v.get('reason','')[:100]}")


## 7 — Emit

Programs are written out **verbatim**, exactly as generated. There is no text
rewriting between what was validated and what ships — that bridge is what produced
the broken llama3b library.

In [ ]:
ok_heads = {k: v for k, v in manifest.items() if v.get("status") == "ok"}
programs = {(v["layer"], v["head"]): (PROG_DIR / f"{k}.py").read_text()
            for k, v in ok_heads.items()}

OUT = ROOT / "data" / f"{MODEL_KEY}_programs.py"
ps.emit_module(OUT, MODEL_KEY, MODEL_ID, N_LAYERS, N_HEADS, programs)
print(f"wrote {OUT}  ({len(programs)}/{N_PROGS} programs, {OUT.stat().st_size/1024:.0f} KB)")

import collections
print("winners:", dict(collections.Counter(v["winner"] for v in ok_heads.values())))

## 8 — Verify

Import the emitted module and actually run every program. This **fails loudly** by
design: every downstream consumer wraps program calls in a bare `except Exception`,
so a broken program silently reads as "head not covered". That is precisely how the
llama3b library shipped 51% broken without anyone noticing.

In [ ]:
report, emitted = ps.verify_module(OUT, tokenizer, score_sentences[:3], N_PROGS)

print(f"programs in module : {report['n_programs']} (expected {report['expected']})")
print(f"run clean          : {report['n_ok']}")
print(f"raised             : {len(report['failures'])}")
print(f"not row-stochastic : {len(report['not_stochastic'])}")
print(f"not causal         : {len(report['not_causal'])}")
for n, e in list(report["failures"].items())[:10]:
    print(f"   {n}: {e}")

assert not report["failures"],       f"{len(report['failures'])} programs raise"
assert not report["not_stochastic"], f"{len(report['not_stochastic'])} not row-stochastic"
assert not report["not_causal"],     f"{len(report['not_causal'])} not causal"
print("\nAll programs execute cleanly.")